# Populacja budynków z ortofotomapy i BDOT10k: proof of concept

Arkusz 6.148.12.04.2, okolice pl. Grunwaldzkiego, 800 × 500 m. Notatnik działa bez klucza API: świadek wizualny pracuje tu na sucho.

In [ ]:
%config InlineBackend.figure_formats = ["jpeg"]
%config InlineBackend.print_figure_kwargs = {"bbox_inches": "tight", "pil_kwargs": {"quality": 88}}

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch, Patch
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
from shapely.geometry import box

from agenci import EKSPERYMENTY
from agenci.eksperyment import uruchom, wczytaj_konfiguracje
from agenci.estymatory import objetosc_bdot
from agenci.karta import zbuduj_karte
from agenci.swiadkowie import tabela
from ortofoto.arkusze import czytaj, zasieg
from ortofoto.budynki import wczytaj_budynki, wczytaj_osm, wybierz

plt.rcParams.update({"figure.dpi": 100, "font.size": 9, "axes.titlesize": 10,
                     "axes.spines.top": False, "axes.spines.right": False})
KOLORY = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]  # kategorie, zawsze w tej kolejności
NIEBIESKI = LinearSegmentedColormap.from_list("niebieski", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
TUSZ, SZARY = "#333333", "#b5b5b5"


def skala(ax, metry, loc="lower right"):
    pasek = AnchoredSizeBar(ax.transData, metry, f"{metry} m", loc, pad=0.4, borderpad=0.5, sep=3,
                            frameon=True, size_vertical=metry / 30)
    pasek.patch.set_alpha(0.8)
    ax.add_artist(pasek)


def podklad(bounds, res):
    """Rozjaśniony, szary wycinek arkusza N: tło, na którym kolory obrysów są czytelne."""
    img, ext = czytaj("N", bounds, res)
    return 255 - (255 - img.mean(2)) * 0.45, ext


bud = wczytaj_budynki()
osm = wczytaj_osm()
ARKUSZ = zasieg().bounds

## 1. Dane na jednym arkuszu

Ortofotomapa skośna z czterech kierunków, BDOT10k (obrysy z ewidencji, funkcja, kondygnacje, stan) i OSM (adres, czasem zmierzona wysokość).

In [ ]:
GRUPY = {"budynki mieszkalne": "mieszkalne", "budynki transportu i łączności": "garaże i transport",
         "budynki oświaty, nauki i kultury oraz budynki sportowe": "oświata i kultura",
         "budynki biurowe": "biura, handel, usługi", "budynki handlowo-usługowe": "biura, handel, usługi",
         "budynki szpitali i inne budynki opieki zdrowotnej": "biura, handel, usługi"}
KOLEJNOSC_GRUP = ["mieszkalne", "garaże i transport", "oświata i kultura", "biura, handel, usługi", "pozostałe"]
bud["grupa"] = bud.funkcja_ogolna.map(GRUPY).fillna("pozostałe")

fig, axs = plt.subplot_mosaic([["mapa", "funkcje"], ["mapa", "pokrycie"]], figsize=(15, 6.4), layout="constrained",
                              width_ratios=[2.3, 1])
img, ext = podklad(ARKUSZ, 1.0)
ax = axs["mapa"]
ax.imshow(img, extent=ext, cmap="gray", vmin=0, vmax=255)
for grupa, kolor in zip(KOLEJNOSC_GRUP, KOLORY):
    bud[bud.grupa == grupa].plot(ax=ax, color=kolor, edgecolor="white", linewidth=0.4)
ax.legend(handles=[Patch(color=k, label=f"{g} ({(bud.grupa == g).sum()})") for g, k in zip(KOLEJNOSC_GRUP, KOLORY)],
          title="funkcja ogólna", loc="upper left", fontsize=8, framealpha=0.9)
ax.set(title=f"{len(bud)} budynków z BDOT10k", xlim=ext[:2], ylim=ext[2:])
ax.set_axis_off()
skala(ax, 100)

funkcje = bud.funkcja.value_counts().head(8)[::-1]
pokrycie = pd.Series({"funkcja (BDOT10k)": bud.funkcja.notna().mean(),
                      "kondygnacje (BDOT10k)": bud.kondygnacje.notna().mean(),
                      "dopasowany obrys OSM": bud.osm_id.notna().mean(),
                      "adres (OSM)": bud["addr:street"].notna().mean(),
                      "zmierzona wysokość (OSM)": bud.h_osm.notna().mean()})[::-1] * 100
for nazwa, dane, tytul, format_ in [("funkcje", funkcje, "Przeważająca funkcja (BDOT10k)", "{:.0f}"),
                                    ("pokrycie", pokrycie, "Ile budynków ma daną informację [%]", "{:.0f}%")]:
    a = axs[nazwa]
    a.barh(dane.index, dane.values, color=KOLORY[0], height=0.65)
    for y, v in enumerate(dane.values):
        a.text(v, y, " " + format_.format(v), va="center", fontsize=8, color=TUSZ)
    a.set_title(tytul)
    a.tick_params(axis="y", length=0)
    a.set_xticks([])
    a.spines[["bottom", "left"]].set_visible(False)
plt.show()

## 2. BDOT10k i OSM razem

Ewidencja dzieli bloki na segmenty, a OSM często ma jeden obrys całego bloku. Segment dostaje z OSM adres i wysokość, gdy obrys OSM pokrywa co najmniej połowę jego powierzchni.

In [ ]:
def zoom(ax, b, r, tytul, podpisy):
    cx, cy = b.geometry.centroid.coords[0]
    okno = (cx - r, cy - 0.7 * r, cx + r, cy + 0.7 * r)
    img, ext = podklad(okno, 0.1)
    ax.imshow(img, extent=ext, cmap="gray", vmin=0, vmax=255)
    w_oknie = box(*okno)
    osm[osm.intersects(w_oknie)].boundary.plot(ax=ax, color=KOLORY[1], linewidth=2.2, linestyle="--")
    bud[bud.intersects(w_oknie)].boundary.plot(ax=ax, color=KOLORY[0], linewidth=1.4)
    for seg in podpisy.itertuples():
        x, y = seg.geometry.representative_point().coords[0]
        if not w_oknie.contains(seg.geometry.representative_point()):
            continue
        ax.text(x, y, seg.podpis, ha="center", va="center", fontsize=7.5, color=TUSZ,
                bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="none", alpha=0.85))
    ax.set(title=tytul, xlim=ext[:2], ylim=ext[2:])
    ax.set_axis_off()
    skala(ax, 10)


blok = bud[bud.osm_id == 101189011].copy()
blok["podpis"] = [f"h = {h:.1f} m" for h in blok.h]
kamienica = wybierz(bud, adres="Piastowska 36")
sasiedzi = bud[bud.intersects(kamienica.geometry.buffer(1))].copy()
sasiedzi["podpis"] = sasiedzi.funkcje.str.replace("budynek wielorodzinny", "wielorodzinny").str.replace("; ", "\n+ ")

fig, axs = plt.subplots(1, 2, figsize=(15, 5.2), layout="constrained")
zoom(axs[0], blok.iloc[1], 32, "Blok: jeden obrys w OSM, trzy segmenty w BDOT10k", blok)
zoom(axs[1], kamienica, 30, "Kamienice przy Piastowskiej: funkcje z BDOT10k", sasiedzi)
fig.legend(handles=[Line2D([], [], color=KOLORY[0], lw=1.6, label="BDOT10k"),
                    Line2D([], [], color=KOLORY[1], lw=2.2, ls="--", label="OSM")], loc="upper center",
           bbox_to_anchor=(0.5, -0.01), ncols=2, frameon=False)
plt.show()

## 3. Karta budynku

Deterministyczna część systemu: wszystko, co model dostaje o jednym budynku. Żółty obrys to podstawa, czerwony to dach przesunięty o 0,85 · h.

In [ ]:
karta = zbuduj_karte(kamienica)
fig, axs = plt.subplot_mosaic([["N", "E", "S", "W", "rekord"], ["fN", "fE", "fS", "fW", "rekord"]], figsize=(17, 7),
                              layout="constrained", width_ratios=[1, 1, 1, 1, 0.9])
for kier in "NESW":
    axs[kier].imshow(karta.widoki[kier])
    axs[kier].set_title(f"widok {kier}")
    if kier in karta.fasady:
        axs["f" + kier].imshow(karta.fasady[kier])
    axs["f" + kier].set_title(f"fasada z widoku {kier}")
    for a in (axs[kier], axs["f" + kier]):
        a.set_axis_off()
axs["rekord"].text(0, 1, tabela(karta), va="top", fontsize=10, family="monospace", wrap=True)
axs["rekord"].set_title("tekst w prompcie", loc="left")
axs["rekord"].set_axis_off()
plt.show()

## 4. Architektura

Świadkowie opisują budynek, a liczbę mieszkańców liczy estymator ze wzoru. Wzór nie potrzebuje sumy rejonu, więc działa też dla osiedli oddanych po 2021 roku. Agenci rozmawiają tylko w pętli uzgadniania: agent bilansujący porównuje predykcje z sumami z 2021 i dopytuje świadka wizualnego oraz agenta rubryki.

Dane statystyczne wchodzą do systemu jako wejście (świadek statystyczny). Woda z MPWiK zostaje wyłącznie do oceny, żeby system nie mógł się do niej dostroić.

In [ ]:
TYP = {"VLM": KOLORY[0], "LLM": "#6250d6", "deterministyczny": "#6b6b6b"}
ROZMOWA = KOLORY[1]
W, H = 18, 10
BLOKI = {  # nazwa: (x, y, tytuł, opis, typ albo None dla danych, działa?)
    "orto": (0, 46, "Ortofotomapa skośna", "N · E · S · W, piksel 5 cm", None, True),
    "bdot": (0, 32, "BDOT10k + OSM", "obrys, funkcja, kondygnacje,\nstan, adres", None, True),
    "gus": (0, 18, "GUS NSP 2021", "ludność w kwadratach\n125 × 125 m", None, True),
    "obwody": (0, 4, "Obwody wyborcze 2025", "mieszkańcy i wyborcy;\nparser granic gotowy", None, False),
    "przestrzenny": (23, 39, "Świadek przestrzenny", "karta budynku: wycinki,\nfasady, rekord BDOT", "deterministyczny", True),
    "statystyczny": (23, 11, "Świadek statystyczny", "ludność kwadratu z 2021\ni udział budynku", "deterministyczny", True),
    "wizualny": (46, 46, "Świadek wizualny", "typ zabudowy, kondygnacje,\nudział mieszkalny, stan", "VLM", True),
    "rubryka": (46, 32, "Agent rubryki", "gęstość dla typu zabudowy;\nteraz z wiedzy modelu", "LLM", True),
    "bilans": (46, 18, "Agent bilansujący", "porównuje z sumą 2021,\nwskazuje nowe osiedla", "LLM", False),
    "estymator": (69, 32, "Estymator", "pow. × kondygnacje ×\nudział × gęstość[typ]", "deterministyczny", True),
    "odniesienia": (69, 11, "Punkty odniesienia", "objętość × BDOT, NSP 2021;\njedno wywołanie VLM (L0)", None, True),
    "kontrola": (92, 46, "Kontrola z BDOT10k", "kondygnacje, mieszkalny", None, True),
    "woda": (92, 32, "Woda MPWiK", "tylko do oceny", None, False),
    "ocena": (92, 18, "Obwody, NSP 2021", "do oceny, jeśli nie są\nwejściem", None, False),
}
PRZEPLYW = [("orto", "przestrzenny"), ("bdot", "przestrzenny"), ("gus", "statystyczny"), ("obwody", "statystyczny"),
            ("przestrzenny", "wizualny"), ("wizualny", "estymator"), ("rubryka", "estymator"),
            ("statystyczny", "bilans"), ("statystyczny", "rubryka"), ("estymator", "bilans"),
            ("estymator", "kontrola"), ("estymator", "woda"), ("estymator", "ocena")]


def kotwica(nazwa, strona):
    x, y, *_ = BLOKI[nazwa]
    return {"prawo": (x + W + 0.5, y + H / 2), "lewo": (x - 0.5, y + H / 2),
            "gora": (x + W / 2, y + H + 0.5), "dol": (x + W / 2, y - 0.5)}[strona]


fig, ax = plt.subplots(figsize=(16, 7.4))
ax.add_patch(FancyBboxPatch((44.2, 16.2), 21.6, 42.4, boxstyle="round,pad=0,rounding_size=2",
                            fc="#fdf1ec", ec=ROZMOWA, lw=1.2, ls="--", zorder=0))
ax.text(55, 59.6, "pętla uzgadniania: tu agenci rozmawiają", ha="center", fontsize=9, color=ROZMOWA)
for nazwa, (x, y, tytul, opis, typ, dziala) in BLOKI.items():
    ax.add_patch(FancyBboxPatch((x, y), W, H, boxstyle="round,pad=0.4,rounding_size=1.2", zorder=2,
                                fc="#f4f7fb" if dziala else "white", ec="#555555" if dziala else "#9a9a9a",
                                lw=1.1, ls="-" if dziala else "--"))
    ax.text(x + W / 2, y + H - 2.6, tytul, ha="center", fontsize=9, weight="bold", color=TUSZ, zorder=3)
    ax.text(x + W / 2, y + H - 4.4, opis, ha="center", va="top", fontsize=8, color=TUSZ, zorder=3)
    if typ:
        ax.text(x + W - 0.2, y + H + 0.2, typ, ha="right", va="center", fontsize=7, color="white", zorder=4,
                bbox=dict(boxstyle="round,pad=0.25", fc=TYP[typ], ec="none"))
for a, b in PRZEPLYW:
    xa, ya, xb, yb = BLOKI[a][0], BLOKI[a][1], BLOKI[b][0], BLOKI[b][1]
    if xa == xb:
        start, koniec = kotwica(a, "dol" if yb < ya else "gora"), kotwica(b, "gora" if yb < ya else "dol")
    elif xb < xa:
        start, koniec = kotwica(a, "lewo"), kotwica(b, "prawo")
    else:
        start, koniec = kotwica(a, "prawo"), kotwica(b, "lewo")
    gotowe = BLOKI[a][5] and BLOKI[b][5]
    ax.annotate("", xy=koniec, xytext=start, zorder=1,
                arrowprops=dict(arrowstyle="-|>", color="#8a8a8a", lw=1, ls="-" if gotowe else "--"))
ROZMOWY = [(kotwica("bilans", "gora"), kotwica("rubryka", "dol")),  # agent bilansujący i agent rubryki
           ((65.2, 23), (65.2, 51))]  # agent bilansujący i świadek wizualny, obok bloków
for start, koniec in ROZMOWY:
    ax.annotate("", xy=koniec, xytext=start, zorder=3,
                arrowprops=dict(arrowstyle="<|-|>", color=ROZMOWA, lw=1.8))
ax.legend(handles=[Patch(fc=TYP["VLM"], label="VLM: patrzy na obrazy"), Patch(fc=TYP["LLM"], label="LLM: rozumuje na tekście"),
                   Patch(fc=TYP["deterministyczny"], label="deterministyczny: bez modelu"),
                   Patch(fc="#f4f7fb", ec="#555555", label="działa"), Patch(fc="white", ec="#9a9a9a", ls="--", label="w planie"),
                   Line2D([], [], color=ROZMOWA, lw=1.8, marker=">", label="rozmowa agentów")],
          loc="upper center", bbox_to_anchor=(0.5, 0.02), ncols=6, frameon=False, fontsize=8.5)
ax.set(xlim=(-1.5, 111), ylim=(2.5, 61))
ax.set_axis_off()
plt.show()

Jedna runda rozmowy w pętli uzgadniania. To plan, liczby są przykładowe.

In [ ]:
UCZESTNICY = [("Świadek statystyczny", "deterministyczny"), ("Agent bilansujący", "LLM"),
              ("Świadek wizualny", "VLM"), ("Agent rubryki", "LLM")]
WIADOMOSCI = [(0, 1, "kwadrat 250 m: NSP 2021 = 640 osób, suma predykcji = 910"),
              (1, 2, "budynki A i B dają razem 300 osób. Czy stały tu w 2021?"),
              (2, 1, "A i B to nowe bloki, zamieszkane; reszta bez zmian"),
              (1, 3, "bez A i B wychodzi 610 wobec 640. Czy gęstość kamienic nie jest za niska?"),
              (3, 1, "kamienica: 2,4 → 2,5 osoby na 100 m²")]

fig, ax = plt.subplots(figsize=(15, 5.6))
for i, (nazwa, typ) in enumerate(UCZESTNICY):
    ax.text(i, 0.35, nazwa, ha="center", fontsize=9.5, weight="bold", color=TUSZ)
    ax.text(i, -0.05, typ, ha="center", fontsize=7.5, color="white",
            bbox=dict(boxstyle="round,pad=0.3", fc=TYP[typ], ec="none"))
    ax.plot([i, i], [-0.4, -len(WIADOMOSCI) - 1.4], color="#c8c8c8", lw=1, ls=(0, (4, 3)), zorder=0)
for n, (a, b, tekst) in enumerate(WIADOMOSCI, start=1):
    y = -n
    kolor = ROZMOWA if 0 not in (a, b) else "#8a8a8a"
    ax.annotate("", xy=(b, y), xytext=(a, y), arrowprops=dict(arrowstyle="-|>", color=kolor, lw=1.6))
    ax.text((a + b) / 2, y + 0.12, tekst, ha="center", va="bottom", fontsize=8.5, color=TUSZ)
ax.text(1, -len(WIADOMOSCI) - 1, "wynik: stare budynki zgadzają się z 2021,\nnowe osiedle dokłada 300 osób ponad sumę z 2021",
        ha="center", va="center", fontsize=8.5, color=TUSZ, bbox=dict(boxstyle="round,pad=0.5", fc="#fdf1ec", ec=ROZMOWA))
ax.set(xlim=(-0.6, 3.6), ylim=(-len(WIADOMOSCI) - 1.8, 0.7))
ax.set_axis_off()
plt.show()

## 5. Ablacje

Każda konfiguracja w `eksperymenty/` zmienia jedną rzecz. Wszystkie dostają tę samą próbkę 40 budynków.

In [ ]:
KOLEJNOSC_KONF = ["objetosc", "objetosc_bdot", "nsp_2021", "wprost_s2", "swiadek_s0", "swiadek_s2", "swiadek_s2_bez_fasad",
                  "swiadek_s2_bez_tabeli", "swiadek_s2_placebo", "sonda_pamieci"]
konfiguracje = [wczytaj_konfiguracje(EKSPERYMENTY / f"{n}.toml") for n in KOLEJNOSC_KONF]
def cechy(k):
    model = k.estymator in ("wprost", "wzor")
    zrodla = set(k.zrodla) if model else set()  # baseline'y nie wysyłają niczego do modelu
    return {"model": model, "rekord BDOT": "tabela" in zrodla or k.estymator in ("objetosc_bdot", "nsp_2021"),
            "NSP 2021": k.estymator == "nsp_2021",
            "widoki": "widoki" in zrodla, "fasady": "fasady" in zrodla, "adres": "adres" in zrodla,
            "placebo": k.placebo, "liczba ze wzoru": k.estymator == "wzor"}


fig, ax = plt.subplots(figsize=(15, 5))
for i, k in enumerate(konfiguracje):
    for j, tak in enumerate(cechy(k).values()):
        ax.scatter(j, i, s=110 if tak else 18, color=KOLORY[0] if tak else "#dedede", zorder=3)
    ax.text(len(cechy(k)) - 0.3, i, k.opis, va="center", fontsize=8.5, color=TUSZ)
CECHY = list(cechy(konfiguracje[0]))
ax.set_xticks(range(len(CECHY)), CECHY, rotation=0)
ax.xaxis.tick_top()
ax.set_yticks(range(len(konfiguracje)), [k.nazwa for k in konfiguracje])
ax.set(xlim=(-0.6, len(CECHY) + 5.5), ylim=(len(konfiguracje) - 0.5, -0.5))
ax.tick_params(length=0)
ax.spines[:].set_visible(False)
ax.grid(axis="y", color="#f0f0f0", zorder=0)
plt.show()

## 6. Przebieg na sucho

Na sucho model nie jest wołany: odpowiedzi to atrapa zgodna ze schematem, a koszt to szacunek z liczby tokenów (Claude Opus 5.5). Prawdziwy przebieg wymaga tylko klucza `ANTHROPIC_API_KEY`.

In [ ]:
z_modelem = [k for k in konfiguracje if k.estymator in ("wprost", "wzor")]
koszt = pd.Series({k.nazwa: uruchom(k, na_sucho=True).koszt_usd.mean() for k in z_modelem}).sort_values()

fig, ax = plt.subplots(figsize=(11, 3.8), layout="constrained")
ax.barh(koszt.index, koszt.values * 100, color=KOLORY[0], height=0.6)
for y, v in enumerate(koszt.values):
    ax.text(v * 100, y, f"  {v * 100:.1f} ¢   (40 budynków ≈ {v * 40:.2f} USD)", va="center", fontsize=8, color=TUSZ)
ax.set(title=f"Szacowany koszt na budynek; cały arkusz ({len(bud)} budynków) ≈ {koszt.max() * len(bud):.0f} USD",
       xlabel="centy USD na budynek")
ax.tick_params(axis="y", length=0)
ax.set_xlim(0, koszt.max() * 160)
plt.show()

## 7. Baseline

Bez modelu: objętość budynków, które BDOT10k uznaje za mieszkalne i eksploatowane. Udział budynku w sumie to jego udział w populacji arkusza, gdyby gęstość wszędzie była taka sama.

In [ ]:
bud["objetosc_bdot"] = [objetosc_bdot(zbuduj_karte(b, widoki=False, fasady=False)) for _, b in bud.iterrows()]
bud["udzial"] = bud.objetosc_bdot / bud.objetosc_bdot.sum() * 100
gora = np.percentile(bud.udzial[bud.udzial > 0], 98)

fig, ax = plt.subplots(figsize=(14, 6.2), layout="constrained")
img, ext = podklad(ARKUSZ, 1.0)
ax.imshow(img, extent=ext, cmap="gray", vmin=0, vmax=255)
bud[bud.udzial == 0].plot(ax=ax, color="#d9d9d9", edgecolor="white", linewidth=0.3)
bud[bud.udzial > 0].plot(ax=ax, column="udzial", cmap=NIEBIESKI, vmin=0, vmax=gora, edgecolor="white", linewidth=0.3,
                         legend=True, legend_kwds={"label": "udział w objętości mieszkalnej arkusza [%]\n"
                                                            "(skala ucięta na 98. percentylu)", "shrink": 0.75})
ax.legend(handles=[Patch(color="#d9d9d9", label="niemieszkalne albo nieeksploatowane")], loc="upper left", fontsize=8)
ax.set(title="objetosc_bdot: rozkład populacji bez modelu", xlim=ext[:2], ylim=ext[2:])
ax.set_axis_off()
skala(ax, 100)
plt.show()

## Co dalej

- Pierwszy prawdziwy przebieg na 40 budynkach i porównanie z `objetosc_bdot`.
- Dane referencyjne: zużycie wody z MPWiK i obwody wyborcze, do tego punkty adresowe miasta. Funkcje oceny już na nie czekają.
- Rubryka skalibrowana na sumach z 2021 roku, potem osiedla oddane po 2021.